In [1]:
list.files('.', pattern='rnaseq-sample-qc[.]csv$', recursive=TRUE, full.names=TRUE)

character(0)

In [2]:
input_path <- "inputs/GSE60450_Lactation-GenewiseCounts-128d2411f316.txt"
source_url <- "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60450"
dat <- read.delim(input_path, header=TRUE, check.names=FALSE, stringsAsFactors=FALSE,
                  quote="", comment.char="", na.strings=c("NA",""))

stopifnot(nrow(dat) == 27179L)
stopifnot(ncol(dat) == 14L)
stopifnot(identical(names(dat)[1:2], c("EntrezGeneID", "Length")))
stopifnot(!anyDuplicated(dat$EntrezGeneID))
stopifnot(!anyNA(dat$EntrezGeneID), !anyNA(dat$Length))

count_names <- names(dat)[-(1:2)]
stopifnot(length(count_names) == 12L)
counts <- as.matrix(dat[, count_names, drop=FALSE])
storage.mode(counts) <- "numeric"
stopifnot(!anyNA(counts))
stopifnot(all(is.finite(counts)))
stopifnot(all(counts >= 0))
stopifnot(all(counts == floor(counts)))

labels <- sprintf("S%02d", seq_along(count_names))
totals <- colSums(counts)
zero_genes <- colSums(counts == 0)
detected <- colSums(counts > 0)
med_detected <- vapply(seq_len(ncol(counts)), function(j) median(counts[counts[,j] > 0, j]), numeric(1))

qc <- data.frame(
  sample_label=labels,
  original_sample_name=count_names,
  total_raw_counts=as.numeric(totals),
  zero_count_genes=as.integer(zero_genes),
  detected_genes=as.integer(detected),
  median_detected_count=as.numeric(med_detected),
  stringsAsFactors=FALSE,
  check.names=FALSE
)
write.csv(qc, "rnaseq-r-sample-qc.csv", row.names=FALSE, quote=TRUE)

png("rnaseq-r-library-sizes.png", width=1800, height=1100, res=180)
op <- par(mar=c(8,6,4,2)+0.1)
bp <- barplot(totals, names.arg=labels, las=2, col="#4472C4", border=NA,
              ylab="Total raw counts", main="GSE60450 raw library-size totals")
text(bp, totals, labels=format(totals, big.mark=",", scientific=FALSE, trim=TRUE),
     pos=3, cex=0.65, srt=45, xpd=TRUE)
mtext("Raw count totals; not normalized expression", side=3, line=0.25, cex=0.9)
par(op)
dev.off()

rver <- R.version.string
mapping_lines <- paste0("| ", labels, " | `", gsub("\\|", "\\\\|", count_names), "` |")
report <- c(
  "# GSE60450 R raw-count QC report",
  "",
  paste0("- Source: ", source_url),
  paste0("- Managed input: `", input_path, "`"),
  paste0("- Runtime: ", rver),
  "- Libraries: base R only; no additional packages installed",
  "- Interpretation: descriptive raw-count quality control only; not differential expression",
  "",
  "## Validation",
  "",
  "- Gene rows: 27,179",
  "- Metadata columns preserved: EntrezGeneID and Length",
  "- Sample-count columns: 12",
  "- EntrezGeneID values unique: yes",
  "- Missing count values: none",
  "- Counts nonnegative and integer-valued: yes",
  "",
  "## Sample metrics",
  "",
  "| Label | Original sample column | Total raw counts | Zero-count genes | Detected genes (>0) | Median among detected genes |",
  "|---|---|---:|---:|---:|---:|",
  vapply(seq_len(nrow(qc)), function(i) paste0("| ", qc$sample_label[i], " | `",
    gsub("\\|", "\\\\|", qc$original_sample_name[i]), "` | ",
    format(qc$total_raw_counts[i], scientific=FALSE, trim=TRUE), " | ",
    qc$zero_count_genes[i], " | ", qc$detected_genes[i], " | ",
    format(qc$median_detected_count[i], scientific=FALSE, trim=TRUE), " |"), character(1)),
  "",
  "## Comparison with Python QC",
  "",
  "The managed artifact catalog lists `rnaseq-sample-qc.csv`, but that artifact was not mounted in the R Notebook and its managed Version ID could not be bound as a Notebook provenance input. Therefore, the requested 48 value-by-value comparisons were not executed, and no match is claimed.",
  "",
  "## Label mapping",
  "",
  "| Compact label | Exact original sample column |",
  "|---|---|",
  mapping_lines,
  "",
  "## Limitations",
  "",
  "These checks describe matrix structure and per-sample raw-count distributions only. Library-size differences can reflect sequencing depth and biology. No normalization, modeling, hypothesis testing, differential-expression analysis, statistical-significance claim, or clinical interpretation was performed."
)
writeLines(report, "rnaseq-r-qc-report.md", useBytes=TRUE)

qc_back <- read.csv("rnaseq-r-sample-qc.csv", check.names=FALSE, stringsAsFactors=FALSE)
png_info <- file.info("rnaseq-r-library-sizes.png")
report_back <- readLines("rnaseq-r-qc-report.md", warn=FALSE)
stopifnot(nrow(qc_back)==12L, ncol(qc_back)==6L)
stopifnot(isTRUE(png_info$size > 0))
stopifnot(length(report_back) > 20L)
list(
  R_version=rver,
  dimensions=c(genes=nrow(dat), metadata_columns=2L, sample_columns=length(count_names)),
  validation=c(unique_gene_ids=!anyDuplicated(dat$EntrezGeneID), no_missing_counts=!anyNA(counts),
               nonnegative=all(counts>=0), integer_valued=all(counts==floor(counts))),
  qc=qc,
  reopened=c(csv=TRUE,png=TRUE,report=TRUE),
  comparison="not executable: managed Python QC CSV content unavailable to R notebook"
)

quartz_off_screen 
                2 
$R_version
[1] "R version 4.4.3 (2025-02-28)"

$dimensions
           genes metadata_columns   sample_columns 
           27179                2               12 

$validation
  unique_gene_ids no_missing_counts       nonnegative    integer_valued 
             TRUE              TRUE              TRUE              TRUE 

$qc
   sample_label              original_sample_name total_raw_counts
1           S01 MCL1-DG_BC2CTUACXX_ACTTGA_L002_R1         23227641
2           S02 MCL1-DH_BC2CTUACXX_CAGATC_L002_R1         21777891
3           S03 MCL1-DI_BC2CTUACXX_ACAGTG_L002_R1         24100765
4           S04 MCL1-DJ_BC2CTUACXX_CGATGT_L002_R1         22665371
5           S05 MCL1-DK_BC2CTUACXX_TTAGGC_L002_R1         21529331
6           S06 MCL1-DL_BC2CTUACXX_ATCACG_L002_R1         20015386
7           S07 MCL1-LA_BC2CTUACXX_GATCAG_L001_R1         20392113
8           S08 MCL1-LB_BC2CTUACXX_TGACCA_L001_R1         21708152
9           S09 MCL1-LC_BC2CTUAC

[Media omitted from immutable Provenance snapshot]